# AI Public-Investor Monitor — Quick Analysis
Run the pipeline, inspect the scorecard, and chart a selected company.

In [ ]:
import os
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from ai_monitor import build_pipeline

SEC_USER_AGENT = os.getenv('SEC_USER_AGENT', 'REPLACE_WITH_YOUR_NAME your.email@example.com')
outputs = build_pipeline(
    Path('config.yaml'),
    user_agent=SEC_USER_AGENT,
    refresh_sec=True,
    with_snippets=False,
)


In [ ]:
scorecard = outputs['scorecard'].copy()
scorecard[[
    'ticker', 'revenue_growth_yoy', 'capex_growth_yoy',
    'downstream_metric_yoy_growth', 'paid_useful_work_score_base',
    'ai_economic_coverage_base',
    'catchup_status_base', 'cohort_irr_base'
]].sort_values('ticker')


In [ ]:
outputs['ecosystem_summary']


In [ ]:
ticker = 'NVDA'
sec = outputs['sec_quarters']
company = sec[sec['ticker'].eq(ticker)].sort_values('period_end')
ax = company.plot(
    x='period_end',
    y=[c for c in ['revenue_ttm', 'gross_profit_ttm', 'capex_ttm'] if c in company],
    marker='o',
    title=f'{ticker}: TTM financial baseline',
)
ax.set_ylabel('USD')
plt.show()


In [ ]:
coverage = outputs['coverage']
company_coverage = coverage[coverage['ticker'].eq(ticker)].dropna(subset=['ai_economic_coverage'])
if company_coverage.empty:
    print('Add AI-specific inputs to data/manual_ai_metrics.csv to calculate coverage.')
else:
    ax = company_coverage.pivot(index='period_end', columns='scenario', values='ai_economic_coverage').plot(
        marker='o', title=f'{ticker}: AI economic coverage'
    )
    ax.axhline(1.0, linestyle='--')
    ax.axhline(1.25, linestyle='--')
    ax.set_ylabel('Contribution profit / capital charge')
    plt.show()
